# Scraper Comparison: BeautifulSoup vs Scrapy vs Crawl4AI

**Objective:** Compare three major Python scraping libraries for extracting content from modern, potentially JavaScript-heavy webpages for AI workflows.

**The Challenge:** Extract clean, meaningful content from a URL.

**Candidates:**

1.  **BeautifulSoup:** The traditional, manual approach.
2.  **Scrapy:** The robust, industrial-strength framework.
3.  **Crawl4AI:** The modern, AI-first async solution.


In [ ]:
import asyncio
import time
import requests
from bs4 import BeautifulSoup
import scrapy
from scrapy.crawler import CrawlerProcess
from crawl4ai import AsyncWebCrawler

# Target URL (Modern, potentially JS-heavy)
URL = "https://www.python.org"

## 1. BeautifulSoup (The Manual Way)

Requires manual HTTP handling, manual DOM traversal, and manual cleaning. Fails on JS-rendered content without extra drivers.


In [ ]:
def scrape_bs4(url):
    print(f"--- BeautifulSoup: {url} ---")
    start = time.time()
    try:
        # 1. Fetch (Synchronous)
        response = requests.get(url, timeout=10)

        # 2. Parse
        soup = BeautifulSoup(response.content, "html.parser")

        # 3. Manual Cleanup (Removing scripts, styles, navs)
        for tag in soup(["script", "style", "nav", "footer", "aside"]):
            tag.decompose()

        # 4. Text Extraction
        text = " ".join(soup.get_text(separator=" ").split())

        print(f"Extracted {len(text)} chars (Raw Text)")
        return {"time": time.time() - start, "loc": "~15 lines"}
    except Exception as e:
        print(f"Error: {e}")
        return None

## 2. Scrapy (The Framework Way)

Powerful but heavyweight. Requires class definitions, project structure, and specific configurations. JS support requires additional middleware.


In [ ]:
# Global bucket to capture spider output for demo
scrapy_capture = {}


class MinimalSpider(scrapy.Spider):
    name = "minimal"
    start_urls = [URL]
    custom_settings = {"LOG_ENABLED": False}  # Silence logs

    def parse(self, response):
        # CSS Selectors required
        text = " ".join(response.css("body ::text").getall())
        scrapy_capture["content"] = text[:500]  # Preview


def scrape_scrapy(url):
    print(f"--- Scrapy: {url} ---")
    start = time.time()
    try:
        process = CrawlerProcess()
        process.crawl(MinimalSpider)
        process.start()  # Blocks process
        print(f"Extracted content via Spider")
        return {"time": time.time() - start, "loc": "~25 lines"}
    except Exception as e:
        print(f"Scrapy Error (Reactor restart?): {e}")
        return None

## 3. Crawl4AI (The AI-First Way)

Async, handles JavaScript automatically, and produces Markdown optimized for LLMs in one line.


In [ ]:
async def scrape_crawl4ai(url):
    print(f"--- Crawl4AI: {url} ---")
    start = time.time()

    # One-liner for full JS execution + Markdown extraction
    async with AsyncWebCrawler(verbose=False) as crawler:
        result = await crawler.arun(url=url)

    print(f"Extracted {len(result.markdown)} chars (Clean Markdown)")
    print(f"Metadata: {result.metadata.keys()}")
    return {"time": time.time() - start, "loc": "~4 lines"}

## Final Comparison

Comparing the effort and output quality for AI workflows.


In [ ]:
async def compare():
    # Run BS4
    res_bs = scrape_bs4(URL)

    # Run Crawl4AI
    res_c4 = await scrape_crawl4ai(URL)

    # Scrapy is tricky in notebooks due to reactor restarts, so we estimate based on the code above
    # res_scrapy = scrape_scrapy(URL)

    print("\n" + "=" * 70)
    print(f"{'Tool':<15} | {'Lines of Code':<15} | {'Ease of Use':<12} | {'AI-Readiness':<15}")
    print("-" * 70)

    data = [
        ("BeautifulSoup", "~15 (High)", "Medium", "Low (Raw Text)"),
        ("Scrapy", "~25 (High)", "Low", "Low (Raw Text)"),
        ("Crawl4AI", "~4 (Low)", "High", "High (Markdown)"),
    ]

    for tool, loc, ease, ai in data:
        print(f"{tool:<15} | {loc:<15} | {ease:<12} | {ai:<15}")
    print("=" * 70)


await compare()

--- BeautifulSoup: https://www.python.org ---
Extracted 4698 chars (Raw Text)
--- Crawl4AI: https://www.python.org ---


[INIT].... → Crawl4AI 0.7.8 

[FETCH]... ↓ https://www.python.org                                                                               |
✓ | ⏱: 1.00s 

[SCRAPE].. ◆ https://www.python.org                                                                               |
✓ | ⏱: 0.02s 

[COMPLETE] ● https://www.python.org                                                                               |
✓ | ⏱: 1.03s 

Extracted 19611 chars (Clean Markdown)
Metadata: dict_keys(['title', 'description', 'keywords', 'author', 'og:type', 'og:site_name', 'og:title', 'og:description', 'og:image', 'og:image:secure_url', 'og:url'])

Tool            | Lines of Code   | Ease of Use  | AI-Readiness   
----------------------------------------------------------------------
BeautifulSoup   | ~15 (High)      | Medium       | Low (Raw Text) 
Scrapy          | ~25 (High)      | Low          | Low (Raw Text) 
Crawl4AI        | ~4 (Low)        | High         | High (Markdown)
